In [44]:
! pip install segmentation_models_pytorch

In [45]:
# All imports
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import pickle
from os.path import splitext
from os import listdir
import numpy as np
from glob import glob
import torch
from torch.utils.data import Dataset
import logging
from PIL import Image
import tifffile as tiff


import torch
from torch.utils.data import DataLoader, random_split
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
from torch import Tensor
import albumentations as A
from albumentations.pytorch import ToTensorV2

import segmentation_models_pytorch as smp


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [46]:
#Definding the dataset
class BasicDataset(Dataset):
    def __init__(self, imgs_dir, scale=1):
        self.imgs_dir = imgs_dir
        self.scale = scale
        assert 0 < scale <= 1, 'Scale must be between 0 and 1'

        self.ids = [splitext(file)[0] for file in listdir(imgs_dir)
                    if not file.startswith('.')]
        print(f'Creating dataset with {len(self.ids)} examples')

    def __len__(self):
        return len(self.ids)

    @classmethod
    def preprocess(cls, tif_img, scale):
        img_nd = np.array(tif_img)
        if len(img_nd.shape) == 2:
            img_nd = np.expand_dims(img_nd, axis=2)
        img_trans = img_nd.transpose((2, 0, 1))
        if img_trans.max() > 1:
            img_trans = img_trans / 65535
        return img_trans

    def __getitem__(self, i):
        idx = self.ids[i]
        img_file = glob(self.imgs_dir+idx+".tif")
        assert len(img_file) == 1, \
            f'Either no image or multiple images found for the ID {idx}: {img_file}'
        img = tiff.imread(img_file[0])
        img = self.preprocess(img, self.scale)
        return {
            'image': torch.from_numpy(img).type(torch.FloatTensor),
        }


In [47]:
def create_model(in_channels=4, device=device):
    model = smp.DeepLabV3Plus(
        encoder_name='resnet50',
        encoder_weights='imagenet',
        in_channels=3,
        classes=1,
        activation='sigmoid'
    )
    original_conv = model.encoder.conv1
    new_conv = nn.Conv2d(
        in_channels,
        original_conv.out_channels,
        kernel_size=original_conv.kernel_size,
        stride=original_conv.stride,
        padding=original_conv.padding,
        bias=False
    )
    with torch.no_grad():
        new_conv.weight[:, :3, :, :] = original_conv.weight.clone()
        if in_channels > 3:
            new_conv.weight[:, 3:, :, :] = original_conv.weight.mean(dim=1, keepdim=True).clone().repeat(1, in_channels - 3, 1, 1)
    model.encoder.conv1 = new_conv
    model.to(device)
    print(f"Model created with {in_channels} input channels and moved to {device}.")
    return model

In [48]:
dir_img = "/kaggle/input/cloud-masking-dataset/content/train/data/"
dataset = BasicDataset(dir_img)

Creating dataset with 10573 examples


In [49]:
model_path = "/kaggle/input/ghaithmodel/pytorch/default/1/ghaithmodel.pth"
model = create_model(in_channels=4, device=device)
model.load_state_dict(torch.load(model_path, map_location=device))
print(f"Successfully loaded weights from {model_path}")
model.eval()

Model created with 4 input channels and moved to cuda.
Successfully loaded weights from /kaggle/input/ghaithmodel/pytorch/default/1/ghaithmodel.pth


/tmp/ipykernel_31/2185397938.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))


DeepLabV3Plus(
  (encoder): ResNetEncoder(
    (conv1): Conv2d(4, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequentia

In [ ]:
def predict_img(net,
                full_img,
                device,
                scale_factor=1,
                out_threshold=0.5):
    net.eval()
    img = full_img
    img = img.unsqueeze(0)
    img = img.to(device=device)
    with torch.no_grad():
        mask = net(img)
    with torch.no_grad():
        mask = net(img)> out_threshold
    return mask.cpu().squeeze().numpy()

In [ ]:
def rle_encode(mask):
    """
    Encodes a binary mask using Run-Length Encoding (RLE).
    
    Args:
        mask (np.ndarray): 2D binary mask (0s and 1s).
    
    Returns:
        str: RLE-encoded string.
    """
    pixels = mask.flatten(order='F')  # Flatten in column-major order
    pixels = np.concatenate([[0], pixels, [0]])  # Add padding to detect transitions
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1  # Get transition indices
    runs[1::2] -= runs[::2]  # Compute run lengths
    runs[::2] -= 1  # Make it 0-indexed instead of 1-indexed

    return " ".join(map(str, runs))  # Convert to string format

def rle_decode(mask_rle, shape):
    """
    Decodes an RLE-encoded string into a binary mask.
    
    Args:
        mask_rle (str): RLE-encoded string.
        shape (tuple): (height, width) of the output mask.
    
    Returns:
        np.ndarray: Decoded binary mask.
    """
    if not mask_rle:
        return np.zeros(shape, dtype=np.uint8)

    s = list(map(int, mask_rle.split()))
    starts, lengths = s[0::2], s[1::2]  # Separate start positions and lengths

    mask = np.zeros(shape[0] * shape[1], dtype=np.uint8)  # Create a flat mask
    for start, length in zip(starts, lengths):
        mask[start:start + length] = 1  # Fill mask with 1s

    return mask.reshape(shape, order='F')  # Reshape in column-major order


In [ ]:
total_score = 0.0
records = []

for x in range(0,len(dataset)):
    img=dataset[x]['image']
    out=predict_img(model,img,device,out_threshold=0.5)
    records.append({
        'id': str(dataset.ids[x]),
        'segmentation': str(rle_encode(out))
    })

In [ ]:
import pandas as pd
df = pd.DataFrame(records, columns=['id','segmentation'])
df.to_csv('submission.csv', index=False)
print(f"Written {len(df)} rows to submission.csv")

In [ ]:
print(df.head(4))

In [ ]:
df = pd.read_csv('/kaggle/working/submission.csv')

In [ ]:
# ! python "/kaggle/input/run-infer/run_infernece.py" --dataset_path "/kaggle/input/cloud-masking-dataset/content/train/data" --model_path "/kaggle/working/ghaithmodel.pth"